In [1]:
# Install dependencies (add imbalanced-learn)
# %pip install pandas matplotlib scikit-learn imbalanced-learn keras tensorflow --quiet

from features import get_train_test_data

import ctypes  # preloaded by .pth, but good practice
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

data = get_train_test_data("data/bank_data_train.csv", sampler="none", num_impute="mean", cat_handle="none", cat_k=1, threshold_missing_data=0.8)

print(f"x_train shape: {data.x_train.shape}")
print(f"x_test shape: {data.x_test.shape}")
print(f"y_train shape: {data.y_train.shape}")
print(f"y_test shape: {data.y_test.shape}")

# Display first 5 rows of x_train
data.x_train.head()

I0000 00:00:1779050172.841904   28561 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779050172.888015   28561 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779050174.078730   28561 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[SelectKBest chi2] keeping 1/1 cat cols
PACK    386.124732 

Selected: ['PACK']
x_train shape: (284152, 90)
x_test shape: (71038, 90)
y_train shape: (284152,)
y_test shape: (71038,)


,CR_PROD_CNT_IL,AMOUNT_RUB_CLO_PRC,PRC_ACCEPTS_A_EMAIL_LINK,PRC_ACCEPTS_A_POS,PRC_ACCEPTS_A_TK,TURNOVER_DYNAMIC_IL_1M,CNT_TRAN_AUT_TENDENCY1M,SUM_TRAN_AUT_TENDENCY1M,AMOUNT_RUB_SUP_PRC,PRC_ACCEPTS_A_AMOBILE,...,PACK_103,PACK_104,PACK_105,PACK_107,PACK_108,PACK_109,PACK_301,PACK_K01,PACK_M01,PACK_O01
54105,2.061893,-0.018943,0.0,0.0,0.0,-0.044935,-0.997829,-1.489598e+00,1.833618,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
313302,-0.244365,-0.430039,0.0,0.0,0.0,-0.044935,0.000000,-3.523313e-16,-0.513119,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
302648,-0.244365,-0.430039,0.0,0.0,0.0,-0.044935,0.000000,-3.523313e-16,-0.634595,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
350887,2.061893,0.246228,0.0,0.0,0.0,-0.044935,0.000000,-3.523313e-16,0.393918,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41493,-0.244365,-0.385170,0.0,0.0,0.0,-0.044935,3.966221,3.721238e+00,0.638030,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [2]:
# print name of features used for training
print(f"Features used for training: {data.x_train.columns.tolist()}")

Features used for training: ['CR_PROD_CNT_IL', 'AMOUNT_RUB_CLO_PRC', 'PRC_ACCEPTS_A_EMAIL_LINK', 'PRC_ACCEPTS_A_POS', 'PRC_ACCEPTS_A_TK', 'TURNOVER_DYNAMIC_IL_1M', 'CNT_TRAN_AUT_TENDENCY1M', 'SUM_TRAN_AUT_TENDENCY1M', 'AMOUNT_RUB_SUP_PRC', 'PRC_ACCEPTS_A_AMOBILE', 'SUM_TRAN_AUT_TENDENCY3M', 'PRC_ACCEPTS_TK', 'PRC_ACCEPTS_A_MTP', 'REST_DYNAMIC_FDEP_1M', 'CNT_TRAN_AUT_TENDENCY3M', 'CNT_ACCEPTS_TK', 'REST_DYNAMIC_SAVE_3M', 'CR_PROD_CNT_VCU', 'REST_AVG_CUR', 'AMOUNT_RUB_NAS_PRC', 'TRANS_COUNT_SUP_PRC', 'PRC_ACCEPTS_A_ATM', 'PRC_ACCEPTS_MTP', 'TRANS_COUNT_NAS_PRC', 'CNT_ACCEPTS_MTP', 'CR_PROD_CNT_TOVR', 'CR_PROD_CNT_PIL', 'TURNOVER_CC', 'TRANS_COUNT_ATM_PRC', 'AMOUNT_RUB_ATM_PRC', 'TURNOVER_PAYM', 'AGE', 'CNT_TRAN_MED_TENDENCY3M', 'CR_PROD_CNT_CC', 'SUM_TRAN_MED_TENDENCY3M', 'REST_DYNAMIC_FDEP_3M', 'REST_DYNAMIC_IL_1M', 'SUM_TRAN_CLO_TENDENCY3M', 'CR_PROD_CNT_CCFP', 'CNT_TRAN_CLO_TENDENCY3M', 'REST_DYNAMIC_CUR_1M', 'REST_AVG_PAYM', 'LDEAL_GRACE_DAYS_PCT_MED', 'REST_DYNAMIC_CUR_3M', 'CNT_TRA

In [ ]:
from NN_GridSearch import create_model, NotebookCallback, PARAM_GRID, LOSS_FUNCTION, TARGET_AUC
from itertools import product
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import pandas as pd
import tensorflow as tf

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" 

PARAM_GRID["epochs"] = [80]  # for demonstration; increase for better results

BEST_MODEL_PATH = "best_model.keras"

weights = compute_class_weight(class_weight="balanced", classes=np.unique(data.y_train), y=data.y_train)
class_weights = dict(enumerate(weights))

keys   = list(PARAM_GRID.keys())
combos = list(product(*PARAM_GRID.values()))
total  = len(combos)

global_best = {"auc": 0.0}
results     = []

print(f"Grid search: {total} combinations\n")

for run_idx, values in enumerate(combos, 1):
    params = dict(zip(keys, values))
    print(f"\n[{run_idx}/{total}] {params}")

    tf.keras.backend.clear_session()

    model = create_model(data.x_train.shape[1], params["dropout_rate"], params["l2_reg"])
    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=params["learning_rate"]),
        loss      = LOSS_FUNCTION,
        metrics   = [tf.keras.metrics.AUC(name="auc")],
    )

    cb = NotebookCallback(TARGET_AUC, global_best, BEST_MODEL_PATH, params)

    model.fit(
        data.x_train, data.y_train,
        epochs          = params["epochs"],
        batch_size      = params["batch_size"],
        callbacks       = [cb],
        validation_data = (data.x_test, data.y_test),
        class_weight    = class_weights,
        verbose         = 0,
    )

    print(f"  ✔ best val_auc: {cb.best_so_far:.4f}")
    results.append({**params, "val_auc": cb.best_so_far})

    if cb.best_so_far >= TARGET_AUC:
        print(f"\n🎯 Target AUC {TARGET_AUC} reached — stopping.")
        break

df = (pd.DataFrame(results)
        .sort_values("val_auc", ascending=False)
        .reset_index(drop=True))
df.index += 1
print(f"\n{'═'*70}\n{df.to_string()}\n{'═'*70}")

Grid search: 16 combinations


[1/16] {'learning_rate': 0.001, 'dropout_rate': 0.1, 'l2_reg': 1e-09, 'batch_size': 128, 'epochs': 80}


I0000 00:00:1779050179.217296   28561 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5520 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
I0000 00:00:1779050183.250478   28729 service.cc:153] XLA service 0x74c8cc0173b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779050183.250540   28729 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4070 Laptop GPU, Compute Capability 8.9 (Driver: 12.6.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1779050183.325977   28729 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779050183.710094   28729 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1779050183.812687   28729 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_4506__.50
I0000 00:00:1779050183.90533

In [ ]:
# Evaluate model on test set
model = tf.keras.models.load_model(BEST_MODEL_PATH)

test_loss, test_auc = model.evaluate(data.x_test, data.y_test, verbose=0)
print(f"Test AUC: {test_auc:.4f}")